# Demo 1 — Output Variance in Practice

**Non-Determinism in Production: What It Is and How to Measure It**  
Charles Rice · March 2, 2026

---

**This notebook was pre-run before the talk. The output cells below are the real results.**

The same prompt is sent to the same model five times under three conditions:

| Experiment | Temperature | What it represents |
|---|---|---|
| 1 | Default (unset) | What most products actually run |
| 2 | 1.0 (explicit) | Maximum sampling diversity |
| 3 | 0.0 (explicit) | Documented by providers as their most deterministic configuration |

---


---
## Step 0 — Discover available Llama models

Run this cell once to print every model ID containing 'llama' available on your GitHub account.  
Copy the correct ID into `MODEL_ID` in the Setup cell below.

In [1]:
# ── Discover model IDs from GitHub Models catalog ─────────────────────────────
# pip install requests

import os
import requests

headers = {
    "Authorization": f"Bearer {os.environ['GITHUB_TOKEN']}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

resp = requests.get("https://models.github.ai/catalog/models", headers=headers)
resp.raise_for_status()

models = resp.json()

# Print every model whose ID contains 'llama' (case-insensitive)
llama_models = [
    m.get("id", m.get("name", str(m)))
    for m in models
    if "llama" in str(m).lower()
]

print("Available Llama models on GitHub Models:")
for mid in sorted(llama_models):
    print(f"  {mid}")

Available Llama models on GitHub Models:
  meta/llama-3.2-11b-vision-instruct
  meta/llama-3.2-90b-vision-instruct
  meta/llama-3.3-70b-instruct
  meta/llama-4-maverick-17b-128e-instruct-fp8
  meta/llama-4-scout-17b-16e-instruct
  meta/meta-llama-3.1-405b-instruct
  meta/meta-llama-3.1-8b-instruct


---
## Setup

In [2]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# pip install openai

import os
import time
from openai import OpenAI

client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=os.environ["GITHUB_TOKEN"],
)

# ── Paste the correct model ID from Step 0 above ──────────────────────────────
MODEL_ID = "meta/llama-3.3-70b-instruct"

RUNS = 5

PROMPT = (
    "A startup is building a customer support product on top of an LLM. "
    "What is the single most important technical consideration they should "
    "address before launch? Answer in two to three sentences."
)


def run_experiment(label: str, temperature=None) -> list:
    print(f"\n{'=' * 72}")
    print(f"EXPERIMENT: {label}")
    print(f"Temperature: {'not set (API default)' if temperature is None else temperature}")
    print(f"{'=' * 72}")

    responses = []
    for i in range(1, RUNS + 1):
        kwargs = dict(
            model=MODEL_ID,
            messages=[{"role": "user", "content": PROMPT}],
        )
        if temperature is not None:
            kwargs["temperature"] = temperature

        response = client.chat.completions.create(**kwargs)
        text = response.choices[0].message.content.strip()
        responses.append(text)
        print(f"\nRun {i}:")
        print(text)
        print("-" * 72)
        time.sleep(1)

    unique = set(responses)
    print(f"\nUnique responses: {len(unique)} of {RUNS}")
    if len(unique) == RUNS:
        print("Every response was different.")
    elif len(unique) == 1:
        print("Every response was identical.")
    else:
        print(f"{RUNS - len(unique)} response(s) shared wording with at least one other.")
    return responses


print(f"Model : {MODEL_ID}")
print(f"Prompt: {PROMPT}")

Model : meta/llama-3.3-70b-instruct
Prompt: A startup is building a customer support product on top of an LLM. What is the single most important technical consideration they should address before launch? Answer in two to three sentences.


---
## Experiment 1 — Default temperature (unset)

No temperature specified.

In [3]:
results_default = run_experiment("Default temperature (unset)")


EXPERIMENT: Default temperature (unset)
Temperature: not set (API default)

Run 1:
The single most important technical consideration for a startup building a customer support product on top of a Large Language Model (LLM) is ensuring the model's ability to handle sensitive customer data securely and in compliance with relevant regulations, such as GDPR and CCPA. This requires implementing robust data protection measures, including encryption, access controls, and data anonymization. By prioritizing data security, the startup can mitigate potential risks and build trust with its customers.
------------------------------------------------------------------------

Run 2:
The single most important technical consideration for a startup building a customer support product on top of a Large Language Model (LLM) is ensuring the model's reliability and accuracy in handling a wide range of customer inquiries. This involves fine-tuning the LLM to minimize errors, biases, and inconsistencies in i

---
## Experiment 2 — Temperature = 1.0

Temperature 1.0 is the standard default for the OpenAI API and is widely used in production. The model samples freely across its full range of candidates.

In [4]:
results_temp1 = run_experiment("Temperature = 1.0", temperature=1.0)


EXPERIMENT: Temperature = 1.0
Temperature: 1.0

Run 1:
The single most important technical consideration for a startup building a customer support product on top of a Large Language Model (LLM) is ensuring the model's reliability and accuracy in handling a wide range of customer inquiries. This involves fine-tuning the LLM to minimize errors, inconsistencies, and potential biases that could lead to incorrect or unhelpful responses. By addressing this consideration, the startup can provide a trustworthy and effective support experience for its users.
------------------------------------------------------------------------

Run 2:
The single most important technical consideration for a startup building a customer support product on top of a Large Language Model (LLM) is ensuring the model's ability to handle sensitive customer data securely and in compliance with relevant regulations, such as GDPR and HIPAA. This requires implementing robust data protection measures, including encryptio

---
## Experiment 3 — Temperature = 0.0

Temperature 0.0 is documented by providers as their most deterministic configuration. It makes the model favour the single highest-probability token at each step — greedy decoding.

Anthropic's documentation states that even at temperature zero, results **"will not be fully deterministic."**  
OpenAI states the Chat Completions API is "non-deterministic by default" — 
even with a seed parameter set, outputs will only be "mostly" identical, not guaranteed.  
This experiment shows whether that holds in practice.

In [5]:
results_temp0 = run_experiment("Temperature = 0.0", temperature=0.0)


EXPERIMENT: Temperature = 0.0
Temperature: 0.0

Run 1:
The single most important technical consideration for a startup building a customer support product on top of a Large Language Model (LLM) is ensuring the model's ability to handle sensitive customer data securely and in compliance with relevant regulations, such as GDPR and CCPA. This involves implementing robust data encryption, access controls, and anonymization techniques to protect customer information. By prioritizing data security, the startup can mitigate potential risks and build trust with its customers.
------------------------------------------------------------------------

Run 2:
The single most important technical consideration for a startup building a customer support product on top of a Large Language Model (LLM) is ensuring the model's ability to handle sensitive customer data securely and in compliance with relevant regulations, such as GDPR and CCPA. This involves implementing robust data encryption, access con

---
## Summary

In [6]:
experiments = [
    ("Default (unset)",   results_default),
    ("Temperature = 1.0", results_temp1),
    ("Temperature = 0.0", results_temp0),
]

print(f"Model: {MODEL_ID}")
print(f"Runs per experiment: {RUNS}")
print()
print(f"{'Experiment':<22} {'Unique responses':>16}")
print("-" * 40)
for label, results in experiments:
    unique = len(set(results))
    print(f"{label:<22} {unique:>10} of {RUNS}")

print()
print(
    "Temperature controls how the model samples from its probability distribution.\n"
    "At temperature zero it favours the highest-probability token at each step.\n"
    "OpenAI, Anthropic, and Google each confirm in their documentation that\n"
    "variability can persist even then."
)

Model: meta/llama-3.3-70b-instruct
Runs per experiment: 5

Experiment             Unique responses
----------------------------------------
Default (unset)                 5 of 5
Temperature = 1.0               5 of 5
Temperature = 0.0               3 of 5

Temperature controls how the model samples from its probability distribution.
At temperature zero it favours the highest-probability token at each step.
OpenAI, Anthropic, and Google each confirm in their documentation that
variability can persist even then.
